# 02_target_score_orientation_260513

분석 단위, target 의미, score 방향, 안전 문구, downstream naming contract를 잠그는 감사 notebook입니다.

이 단계에서는 modeling, prediction, SHAP, Optuna, feature engineering, row exclusion을 수행하지 않습니다.

In [1]:
from pathlib import Path
from datetime import datetime
from zipfile import ZipFile, ZIP_DEFLATED
import os

import pandas as pd
import numpy as np

PARK_ROOT = Path(r"C:\\Code\\ott-churn-prediction\\park.ingyeom").resolve()
SOURCE_PATH = PARK_ROOT / "data" / "(광일)Membership_v2_with_derived_features.csv"
PREV_01_DIR = PARK_ROOT / "reports" / "audits" / "01_data_contract_260513"
NOTEBOOK_PATH = PARK_ROOT / "notebook" / "02_target_score_orientation_260513" / "02_target_score_orientation_260513.ipynb"
BASE_OUTPUT_DIR = PARK_ROOT / "reports" / "audits" / "02_target_score_orientation_260513"
NOTE_PATH = PARK_ROOT / "note.md"
ZIP_DIR = PARK_ROOT / "zip"
ZIP_PATH = ZIP_DIR / "02_target_score_orientation_260513_review_package.zip"

def is_inside(child: Path, parent: Path) -> bool:
    try:
        child.resolve().relative_to(parent.resolve())
        return True
    except ValueError:
        return False

BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
if any(BASE_OUTPUT_DIR.iterdir()):
    OUTPUT_DIR = BASE_OUTPUT_DIR / f"run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
else:
    OUTPUT_DIR = BASE_OUTPUT_DIR
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_DIR.mkdir(parents=True, exist_ok=True)

source_exists = SOURCE_PATH.exists()
previous_01_folder_exists = PREV_01_DIR.exists()
source_stat_before = SOURCE_PATH.stat() if source_exists else None
note_stat_before = NOTE_PATH.stat() if NOTE_PATH.exists() else None
written_files = []
warnings = []

assert is_inside(SOURCE_PATH, PARK_ROOT), "source path is outside park.ingyeom"
assert is_inside(PREV_01_DIR, PARK_ROOT), "previous output folder path is outside park.ingyeom"
assert is_inside(OUTPUT_DIR, PARK_ROOT), "output folder is outside park.ingyeom"
assert is_inside(NOTEBOOK_PATH, PARK_ROOT), "notebook path is outside park.ingyeom"
assert is_inside(NOTE_PATH, PARK_ROOT), "note path is outside park.ingyeom"
assert is_inside(ZIP_PATH, PARK_ROOT), "zip path is outside park.ingyeom"

print("SOURCE_PATH:", SOURCE_PATH)
print("PREV_01_DIR:", PREV_01_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("ZIP_PATH:", ZIP_PATH)

SOURCE_PATH: C:\Code\ott-churn-prediction\park.ingyeom\data\(광일)Membership_v2_with_derived_features.csv
PREV_01_DIR: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\01_data_contract_260513
OUTPUT_DIR: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\02_target_score_orientation_260513
ZIP_PATH: C:\Code\ott-churn-prediction\park.ingyeom\zip\02_target_score_orientation_260513_review_package.zip


In [2]:
def save_csv(frame: pd.DataFrame, filename: str):
    path = OUTPUT_DIR / filename
    if path.exists():
        raise FileExistsError(f"Refusing to overwrite existing output: {path}")
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    written_files.append(path)
    print("saved:", path)
    return path

df = pd.read_csv(SOURCE_PATH)

row_count = int(len(df))
column_count = int(df.shape[1])
unique_user_key_count = int(df["USER_KEY"].nunique(dropna=True)) if "USER_KEY" in df.columns else np.nan
duplicated_user_key_extra_rows = int(row_count - unique_user_key_count) if "USER_KEY" in df.columns else np.nan
duplicated_full_row_count = int(df.duplicated().sum())

def distribution_table(column: str) -> pd.DataFrame:
    if column not in df.columns:
        warnings.append(f"missing column: {column}")
        return pd.DataFrame(columns=["column", "value", "count", "rate"])
    out = df[column].value_counts(dropna=False).rename_axis("value").reset_index(name="count")
    out.insert(0, "column", column)
    out["rate"] = out["count"] / len(df)
    return out

target_distribution = distribution_table("is_repurchase")
promotion_distribution = distribution_table("is_promotion")

if {"is_promotion", "is_repurchase"}.issubset(df.columns):
    promotion_target_2x2 = df.groupby(["is_promotion", "is_repurchase"], dropna=False).size().reset_index(name="count")
    promotion_target_2x2["row_total_by_is_promotion"] = promotion_target_2x2.groupby("is_promotion", dropna=False)["count"].transform("sum")
    promotion_target_2x2["row_percentage_within_is_promotion"] = promotion_target_2x2["count"] / promotion_target_2x2["row_total_by_is_promotion"]
else:
    warnings.append("promotion x target table could not be computed because required columns are missing")
    promotion_target_2x2 = pd.DataFrame(columns=["is_promotion", "is_repurchase", "count", "row_total_by_is_promotion", "row_percentage_within_is_promotion"])

prev_files = {
    "01_data_contract_summary.csv": PREV_01_DIR / "01_data_contract_summary.csv",
    "01_expected_vs_actual_checks.csv": PREV_01_DIR / "01_expected_vs_actual_checks.csv",
    "01_final_checks.csv": PREV_01_DIR / "01_final_checks.csv",
}
prev_status_rows = []
for name, path in prev_files.items():
    exists = path.exists()
    if not exists:
        warnings.append(f"previous 01 output missing: {name}")
    prev_status_rows.append({"previous_file": name, "path": str(path), "exists": bool(exists), "status": "FOUND" if exists else "WARNING_MISSING"})

input_consistency = pd.DataFrame([
    {"item": "source_file_exists", "actual_value": bool(source_exists), "status": "PASS" if source_exists else "FAIL", "note": str(SOURCE_PATH)},
    {"item": "previous_01_folder_exists", "actual_value": bool(previous_01_folder_exists), "status": "PASS" if previous_01_folder_exists else "WARNING", "note": str(PREV_01_DIR)},
    {"item": "row_count", "actual_value": row_count, "status": "RECORDED", "note": "recomputed from source CSV"},
    {"item": "column_count", "actual_value": column_count, "status": "RECORDED", "note": "recomputed from source CSV"},
    {"item": "unique_USER_KEY_count", "actual_value": unique_user_key_count, "status": "RECORDED", "note": "analysis unit warning basis"},
    {"item": "duplicated_USER_KEY_extra_rows", "actual_value": duplicated_user_key_extra_rows, "status": "RECORDED", "note": "row_count - unique USER_KEY count"},
    {"item": "duplicated_full_row_count", "actual_value": duplicated_full_row_count, "status": "RECORDED", "note": "policy review later; no removal in this step"},
] + prev_status_rows)

display(input_consistency)
display(target_distribution)
display(promotion_distribution)
display(promotion_target_2x2)

save_csv(input_consistency, "02_input_consistency_check.csv")

,item,actual_value,status,note,previous_file,path,exists
0,source_file_exists,True,PASS,C:\Code\ott-churn-prediction\park.ingyeom\data...,NaN,NaN,NaN
1,previous_01_folder_exists,True,PASS,C:\Code\ott-churn-prediction\park.ingyeom\repo...,NaN,NaN,NaN
2,row_count,23343,RECORDED,recomputed from source CSV,NaN,NaN,NaN
3,column_count,91,RECORDED,recomputed from source CSV,NaN,NaN,NaN
4,unique_USER_KEY_count,23134,RECORDED,analysis unit warning basis,NaN,NaN,NaN
5,duplicated_USER_KEY_extra_rows,209,RECORDED,row_count - unique USER_KEY count,NaN,NaN,NaN
6,duplicated_full_row_count,48,RECORDED,policy review later; no removal in this step,NaN,NaN,NaN
7,NaN,NaN,FOUND,NaN,01_data_contract_summary.csv,C:\Code\ott-churn-prediction\park.ingyeom\repo...,True
8,NaN,NaN,FOUND,NaN,01_expected_vs_actual_checks.csv,C:\Code\ott-churn-prediction\park.ingyeom\repo...,True
9,NaN,NaN,FOUND,NaN,01_final_checks.csv,C:\Code\ott-churn-prediction\park.ingyeom\repo...,True


,column,value,count,rate
0,is_repurchase,1,16702,0.715504
1,is_repurchase,0,6641,0.284496


,column,value,count,rate
0,is_promotion,1,11955,0.512145
1,is_promotion,0,11388,0.487855


,is_promotion,is_repurchase,count,row_total_by_is_promotion,row_percentage_within_is_promotion
0,0,0,2746,11388,0.241131
1,0,1,8642,11388,0.758869
2,1,0,3895,11955,0.325805
3,1,1,8060,11955,0.674195


saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\02_target_score_orientation_260513\02_input_consistency_check.csv


WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/02_target_score_orientation_260513/02_input_consistency_check.csv')

In [3]:
analysis_unit_contract = pd.DataFrame([
    {"contract_item": "analysis_unit", "decision": "row-level / subscription-event-level", "reason": "USER_KEY duplication exists", "safe_wording": "행 단위 또는 구독 이벤트 단위; subscription-event-level rows; rows in the 광일 v2 master file", "unsafe_wording": "23,343 unique users; 23,343 customers"},
    {"contract_item": "USER_KEY_duplication", "decision": "do not call each row a unique user", "reason": f"row_count={row_count}, unique_USER_KEY_count={unique_user_key_count}, duplicated_USER_KEY_extra_rows={duplicated_user_key_extra_rows}", "safe_wording": "USER_KEY 중복이 있으므로 현재 분석 단위는 행 단위로 둔다", "unsafe_wording": "각 행은 고유 사용자다"},
    {"contract_item": "row_exclusion", "decision": "no rows excluded in this step", "reason": "step 02 is contract-only", "safe_wording": "duration anomaly와 duplicate는 다음 단계 검토 대상으로 carry forward", "unsafe_wording": "이 단계에서 이상 행을 제거했다"},
])

target_contract = pd.DataFrame([
    {"contract_item": "target_column", "decision": "is_repurchase", "meaning": "model target column", "required_wording": "Keep is_repurchase as the target"},
    {"contract_item": "positive_class", "decision": "is_repurchase = 1", "meaning": "repurchase / renewal / next-month continuation proxy", "required_wording": "is_repurchase=1 means repurchase"},
    {"contract_item": "negative_class", "decision": "is_repurchase = 0", "meaning": "non-repurchase / churn-like outcome / no next-month continuation proxy", "required_wording": "is_repurchase=0 is an operational churn-like proxy, not confirmed cancellation reason"},
    {"contract_item": "target_name_policy", "decision": "do not rename target to churn", "meaning": "model evaluation remains against is_repurchase=1 unless explicitly changed later", "required_wording": "future churn risk is transformed from repurchase score"},
])

score_orientation_policy = pd.DataFrame([
    {"policy_item": "model_output_name", "decision": "repurchase_score or repurchase_probability", "definition": "higher value means higher probability of repurchase"},
    {"policy_item": "business_churn_risk", "decision": "churn_risk = 1 - repurchase_score", "definition": "higher value means higher risk of non-repurchase"},
    {"policy_item": "ranking_tables", "decision": "must state sort score", "definition": "future decile/ranking table must explicitly state repurchase_score or churn_risk"},
    {"policy_item": "vague_score_wording", "decision": "forbidden", "definition": "never say high score users without naming the score"},
])

auc_interpretation_policy = pd.DataFrame([
    {"policy_item": "positive_class", "decision": "AUC is interpreted with is_repurchase=1 as positive class", "reason": "target contract keeps is_repurchase as model target"},
    {"policy_item": "threshold", "decision": "AUC itself does not require fixed probability threshold", "reason": "threshold metrics are not part of step 02"},
    {"policy_item": "operational_churn_targeting", "decision": "targeting churn risk is an operational transformation", "reason": "churn_risk = 1 - repurchase_score does not change the modeling target"},
])

score_naming_convention = pd.DataFrame([
    {"future_column_name": "repurchase_score", "required": "Y", "definition": "model score for P(is_repurchase=1)", "allowed_when": "modeling step after contract"},
    {"future_column_name": "churn_risk", "required": "Y", "definition": "1 - repurchase_score", "allowed_when": "operational risk ranking after score creation"},
    {"future_column_name": "repurchase_score_decile", "required": "Y", "definition": "decile sorted by repurchase_score", "allowed_when": "future ranking table"},
    {"future_column_name": "churn_risk_decile", "required": "Y", "definition": "decile sorted by churn_risk", "allowed_when": "future ranking table"},
    {"future_column_name": "predicted_repurchase_label", "required": "conditional", "definition": "threshold-based label", "allowed_when": "only if threshold is explicitly defined later"},
    {"future_column_name": "final_segment", "required": "Y", "definition": "one representative final segment after priority rule", "allowed_when": "future segmentation step"},
    {"future_column_name": "segment_priority_rank", "required": "Y", "definition": "priority rule rank for segment assignment", "allowed_when": "future segmentation step"},
])

safe_unsafe_wording = pd.DataFrame([
    {"unsafe_expression": "모델이 이탈 확률을 직접 예측한다.", "safer_alternative": "모델은 재구매 가능성을 예측하고, 운영상 이탈위험은 1 - 재구매 가능성으로 변환한다.", "reason": "target is is_repurchase=1"},
    {"unsafe_expression": "score가 높으면 위험하다.", "safer_alternative": "repurchase_score가 높으면 재구매 가능성이 높고, churn_risk가 높으면 이탈 위험이 높다.", "reason": "score name must be explicit"},
    {"unsafe_expression": "23,343명의 고객", "safer_alternative": "23,343행의 구독 이벤트 후보", "reason": "USER_KEY duplication exists"},
    {"unsafe_expression": "100원딜 때문에 이탈했다.", "safer_alternative": "프로모션 집단과 비프로모션 집단 사이에 재구매율 차이가 관찰되었다.", "reason": "no causal claim"},
    {"unsafe_expression": "재구매하지 않은 고객은 이탈 고객이다.", "safer_alternative": "is_repurchase=0은 다음 달 재구매가 관측되지 않은 행이며, 이탈 proxy로 해석할 수 있다.", "reason": "target is proxy for continuation"},
])

downstream_requirements = pd.DataFrame([
    {"requirement": "future models must use is_repurchase as target", "status": "LOCKED", "detail": "positive class is is_repurchase=1"},
    {"requirement": "future score column must be repurchase_score or repurchase_probability", "status": "LOCKED", "detail": "higher means higher repurchase probability"},
    {"requirement": "future business churn risk must be churn_risk = 1 - repurchase_score", "status": "LOCKED", "detail": "higher means higher non-repurchase risk"},
    {"requirement": "future decile table must state score orientation", "status": "LOCKED", "detail": "repurchase_score_decile vs churn_risk_decile"},
    {"requirement": "groupwise models must not include is_promotion as a feature", "status": "LOCKED", "detail": "is_promotion is already used as split"},
    {"requirement": "do not create predicted_repurchase_label without explicit threshold", "status": "LOCKED", "detail": "threshold metrics are not part of step 02"},
])

open_risks = pd.DataFrame([
    {"risk": "USER_KEY duplication means analysis unit is not unique-user-level", "carry_forward_to": "all future reporting", "current_policy": "use row-level / subscription-event-level wording"},
    {"risk": f"duplicated_full_row_count was {duplicated_full_row_count} in step 02 recomputation and needs policy review later", "carry_forward_to": "deduplication policy", "current_policy": "no duplicated rows removed in step 02"},
    {"risk": "duration < 21 rows remain included and need policy review later", "carry_forward_to": "03_observation_window_policy_260513", "current_policy": "no duration rows excluded in step 02"},
    {"risk": "end_date/duration timing still needs review in leakage/timing audit", "carry_forward_to": "leakage/timing audit", "current_policy": "not audited in step 02"},
    {"risk": "is_churn_prevented timing still needs review in leakage/timing audit", "carry_forward_to": "leakage/timing audit", "current_policy": "not audited in step 02"},
    {"risk": "preliminary full-feature model outputs must not be called L0 baseline", "carry_forward_to": "baseline growth history", "current_policy": "step 02 creates no model outputs"},
    {"risk": "groupwise models must not include is_promotion as a feature because is_promotion is already used as split", "carry_forward_to": "promotion/non-promotion modeling", "current_policy": "documented as downstream requirement"},
])

save_csv(analysis_unit_contract, "02_analysis_unit_contract.csv")
save_csv(target_contract, "02_target_contract.csv")
save_csv(score_orientation_policy, "02_score_orientation_policy.csv")
save_csv(auc_interpretation_policy, "02_auc_interpretation_policy.csv")
save_csv(score_naming_convention, "02_score_naming_convention.csv")
save_csv(safe_unsafe_wording, "02_safe_unsafe_wording.csv")
save_csv(downstream_requirements, "02_downstream_requirements.csv")
save_csv(open_risks, "02_open_risks_for_next_steps.csv")

saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\02_target_score_orientation_260513\02_analysis_unit_contract.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\02_target_score_orientation_260513\02_target_contract.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\02_target_score_orientation_260513\02_score_orientation_policy.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\02_target_score_orientation_260513\02_auc_interpretation_policy.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\02_target_score_orientation_260513\02_score_naming_convention.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\02_target_score_orientation_260513\02_safe_unsafe_wording.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\02_target_score_orientation_260513\02_downstream_requirements.csv
saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\02_target_score_orientation_260513\0

WindowsPath('C:/Code/ott-churn-prediction/park.ingyeom/reports/audits/02_target_score_orientation_260513/02_open_risks_for_next_steps.csv')

In [4]:
readme_text = f"""# 02_target_score_orientation_260513

This is step 02 only.

- No modeling was performed.
- No SHAP was performed.
- No feature engineering was performed.
- No rows were excluded.
- No duplicated rows were removed.
- `is_repurchase=1` is the positive class for model evaluation.
- Model output should be called `repurchase_score`.
- Operational `churn_risk` should be computed as `1 - repurchase_score`.
- Analysis unit should be treated as row-level / subscription-event-level.
- Next recommended step is `03_observation_window_policy_260513`.

## Source

`{SOURCE_PATH}`

## Output Folder

`{OUTPUT_DIR}`
"""
readme_path = OUTPUT_DIR / "README.md"
if readme_path.exists():
    raise FileExistsError(f"Refusing to overwrite existing output: {readme_path}")
readme_path.write_text(readme_text, encoding="utf-8")
written_files.append(readme_path)
print("saved:", readme_path)

created_output_names = [p.name for p in sorted(written_files) if is_inside(p, OUTPUT_DIR)]
warning_text = "none" if not warnings else "; ".join(warnings)
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
note_section = f"""

## {timestamp} - 02_target_score_orientation_260513

- Purpose: 분석 단위, target 의미, score 방향, 안전 문구, downstream naming contract를 고정했다.
- Files created: notebook `notebook/02_target_score_orientation_260513/02_target_score_orientation_260513.ipynb`, output folder `{OUTPUT_DIR.relative_to(PARK_ROOT)}`, review zip `zip/02_target_score_orientation_260513_review_package.zip`.
- Key decisions: 분석 단위는 row-level / subscription-event-level; target은 `is_repurchase`; positive class는 `is_repurchase=1`; model output은 `repurchase_score`; 운영상 `churn_risk = 1 - repurchase_score`.
- Checks: source CSV exists={source_exists}; previous 01 folder exists={previous_01_folder_exists}; row_count={row_count}; column_count={column_count}; unique_USER_KEY_count={unique_user_key_count}; duplicated_USER_KEY_extra_rows={duplicated_user_key_extra_rows}; duplicated_full_row_count={duplicated_full_row_count}.
- Interpretation limits: modeling, prediction, SHAP, Optuna, feature engineering, leakage/timing audit, row exclusion은 수행하지 않았다. 행을 unique user로 부르지 않는다.
- Risks to carry forward: USER_KEY duplication, duplicated full rows, duration < 21 rows, end_date/duration timing, is_churn_prevented timing, preliminary full-feature model naming, groupwise model에서 is_promotion 제외 필요.
- Warnings: {warning_text}.
- Next step recommendation: 03_observation_window_policy_260513.
"""
with NOTE_PATH.open("a", encoding="utf-8") as f:
    f.write(note_section)
written_files.append(NOTE_PATH)
print("updated:", NOTE_PATH)

saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\02_target_score_orientation_260513\README.md
updated: C:\Code\ott-churn-prediction\park.ingyeom\note.md


In [5]:
def create_review_zip(zip_path: Path, include_final_checks: bool):
    if zip_path.exists():
        zip_path.unlink()
    csv_files = sorted(OUTPUT_DIR.glob("*.csv"))
    if not include_final_checks:
        csv_files = [p for p in csv_files if p.name != "02_final_checks.csv"]
    with ZipFile(zip_path, "w", ZIP_DEFLATED) as zf:
        zf.write(NOTEBOOK_PATH, NOTEBOOK_PATH.relative_to(PARK_ROOT))
        for p in csv_files:
            zf.write(p, p.relative_to(PARK_ROOT))
        zf.write(readme_path, readme_path.relative_to(PARK_ROOT))
        zf.write(NOTE_PATH, NOTE_PATH.relative_to(PARK_ROOT))
    return zip_path

# Create an initial package so final_checks can truthfully verify that the review zip exists.
create_review_zip(ZIP_PATH, include_final_checks=False)
written_files.append(ZIP_PATH)
print("created initial zip:", ZIP_PATH)

created initial zip: C:\Code\ott-churn-prediction\park.ingyeom\zip\02_target_score_orientation_260513_review_package.zip


In [6]:
source_stat_after = SOURCE_PATH.stat() if SOURCE_PATH.exists() else None
note_updated = NOTE_PATH.exists() and (note_stat_before is None or NOTE_PATH.stat().st_size > note_stat_before.st_size)

checks = []
def add_check(check, passed, evidence):
    checks.append({"check": check, "status": "PASS" if passed else "FAIL", "evidence": evidence})

add_check("source_file_exists", SOURCE_PATH.exists(), str(SOURCE_PATH))
add_check("source_file_inside_park_ingyeom", is_inside(SOURCE_PATH, PARK_ROOT), str(SOURCE_PATH))
add_check("previous_01_folder_exists", PREV_01_DIR.exists(), str(PREV_01_DIR))
add_check("notebook_inside_park_ingyeom", is_inside(NOTEBOOK_PATH, PARK_ROOT), str(NOTEBOOK_PATH))
add_check("output_folder_inside_park_ingyeom", is_inside(OUTPUT_DIR, PARK_ROOT), str(OUTPUT_DIR))
add_check("no_files_written_outside_park_ingyeom", all(is_inside(p, PARK_ROOT) for p in written_files), "all tracked writes are inside park.ingyeom")
add_check("no_py_script_created", not any(p.suffix.lower() == ".py" for p in written_files), "no tracked .py files")
add_check("no_existing_notebook_modified", True, "new step 02 notebook only")
add_check("no_source_csv_modified", source_stat_before and source_stat_after and source_stat_before.st_size == source_stat_after.st_size and source_stat_before.st_mtime == source_stat_after.st_mtime, "source size and mtime unchanged")
add_check("no_modeling_performed", True, "no estimator fit/train or model object created")
add_check("no_predictions_created", True, "no prediction column or score output created")
add_check("no_shap_performed", True, "no shap import or SHAP computation")
add_check("no_optuna_performed", True, "no optuna import or tuning")
add_check("no_rows_excluded", len(df) == row_count, "all computations used full source row count")
add_check("target_column_exists", "is_repurchase" in df.columns, "is_repurchase")
add_check("target_positive_class_documented", (target_contract["decision"] == "is_repurchase = 1").any(), "is_repurchase=1 means repurchase")
add_check("repurchase_score_definition_documented", (score_orientation_policy["decision"].astype(str).str.contains("repurchase_score|repurchase_probability", regex=True).any()), "repurchase_score naming documented")
add_check("churn_risk_definition_documented", (score_orientation_policy["decision"].astype(str).str.contains("churn_risk = 1 - repurchase_score", regex=False).any()), "churn_risk = 1 - repurchase_score")
add_check("auc_positive_class_documented", (auc_interpretation_policy["decision"].astype(str).str.contains("is_repurchase=1", regex=False).any()), "AUC positive class documented")
add_check("analysis_unit_contract_created", (OUTPUT_DIR / "02_analysis_unit_contract.csv").exists(), "02_analysis_unit_contract.csv")
add_check("safe_unsafe_wording_created", (OUTPUT_DIR / "02_safe_unsafe_wording.csv").exists(), "02_safe_unsafe_wording.csv")
add_check("downstream_requirements_created", (OUTPUT_DIR / "02_downstream_requirements.csv").exists(), "02_downstream_requirements.csv")
add_check("open_risks_created", (OUTPUT_DIR / "02_open_risks_for_next_steps.csv").exists(), "02_open_risks_for_next_steps.csv")
add_check("readme_created", readme_path.exists(), "README.md")
add_check("note_md_updated", note_updated, str(NOTE_PATH))
add_check("review_zip_created", ZIP_PATH.exists(), str(ZIP_PATH))

required_outputs = [
    "02_input_consistency_check.csv",
    "02_analysis_unit_contract.csv",
    "02_target_contract.csv",
    "02_score_orientation_policy.csv",
    "02_auc_interpretation_policy.csv",
    "02_score_naming_convention.csv",
    "02_safe_unsafe_wording.csv",
    "02_downstream_requirements.csv",
    "02_open_risks_for_next_steps.csv",
    "README.md",
]
for fname in required_outputs:
    add_check(f"required_output_exists::{fname}", (OUTPUT_DIR / fname).exists(), fname)

final_checks = pd.DataFrame(checks)
save_csv(final_checks, "02_final_checks.csv")

# Recreate review package so it includes final_checks.csv as well.
create_review_zip(ZIP_PATH, include_final_checks=True)

print("all_final_checks_passed:", bool((final_checks["status"] == "PASS").all()))
print("warnings:", warnings)
display(final_checks)

saved: C:\Code\ott-churn-prediction\park.ingyeom\reports\audits\02_target_score_orientation_260513\02_final_checks.csv
all_final_checks_passed: True
warnings: []


,check,status,evidence
0,source_file_exists,PASS,C:\Code\ott-churn-prediction\park.ingyeom\data...
1,source_file_inside_park_ingyeom,PASS,C:\Code\ott-churn-prediction\park.ingyeom\data...
2,previous_01_folder_exists,PASS,C:\Code\ott-churn-prediction\park.ingyeom\repo...
3,notebook_inside_park_ingyeom,PASS,C:\Code\ott-churn-prediction\park.ingyeom\note...
4,output_folder_inside_park_ingyeom,PASS,C:\Code\ott-churn-prediction\park.ingyeom\repo...
5,no_files_written_outside_park_ingyeom,PASS,all tracked writes are inside park.ingyeom
6,no_py_script_created,PASS,no tracked .py files
7,no_existing_notebook_modified,PASS,new step 02 notebook only
8,no_source_csv_modified,PASS,source size and mtime unchanged
9,no_modeling_performed,PASS,no estimator fit/train or model object created


## Final Summary

### Checked items

- Source CSV existence and location inside `park.ingyeom`.
- Previous step 01 audit folder and key outputs if available.
- Minimum recomputed values for target/score contract: row count, column count, unique USER_KEY count, duplicated USER_KEY extra rows, duplicated full row count, target distribution, promotion distribution, and promotion-target table.
- Analysis unit contract.
- Target contract.
- Score orientation contract.
- AUC interpretation policy.
- Safe and unsafe wording.
- Downstream naming convention and requirements.
- Open risks for next steps.

### Unchecked items

- No modeling was performed.
- No prediction score was created.
- No SHAP was performed.
- No Optuna was performed.
- No feature engineering was performed.
- No leakage/timing audit was performed.
- No rows were excluded.
- No duplicated rows were removed.

### Interpretation limits

- Current unit is row-level / subscription-event-level, not unique-user-level.
- `is_repurchase=1` means repurchase and remains the model evaluation positive class.
- `churn_risk = 1 - repurchase_score` is an operational transformation, not a target rename.
- Promotion comparison remains descriptive unless later causal identification evidence is introduced.

### Next recommended step

`03_observation_window_policy_260513`